# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook presents a step-by-step guide for exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library. You will load, inspect, and process the dataset, referencing all key entities such as record sets, fields, and columns by their `@id` values for clarity and reproducibility.

### Dataset Source
The dataset's Croissant schema can be found at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

We'll load both the dataset metadata and the records using `mlcroissant`. This step enables us to inspect and reference the dataset contents throughout our workflow.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview

Let's review all record sets and their available fields. We will reference each by their `@id`, which ensures precision and reproducibility in further analyses.

In [ ]:
# List all record sets by their @id
print("Record Sets (by @id):")
for record_set in metadata.record_sets:
    print(f"  - RecordSet @id: {record_set.id}, Name: {getattr(record_set, 'name', '')}")
    # List all fields in the record set by @id
    print("    Fields and columns (by @id):")
    for field in getattr(record_set, 'fields', []):
        # Each field could contain a column object
        field_desc = f"{field.id}"
        if getattr(field, 'column', None) is not None:
            if isinstance(field.column, (list, tuple)):
                field_desc += " (columns: " + ", ".join([c.id for c in field.column]) + ")"
            else:
                field_desc += f" (column: {field.column.id})"
        print(f"      - Field @id: {field_desc}; Name: {getattr(field, 'name', getattr(field, 'id', ''))}; DataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction

Let's extract data for each record set. We'll convert each record set into a pandas DataFrame. Use `@id` values for all references.

In [ ]:
# Extract data from each record set using their @id
record_sets_ids = [record_set.id for record_set in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"\nLoading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  - Loaded {len(df)} records.")
    print(f"  - Columns (@id): {df.columns.tolist()}")

# For demonstration, pick the first record set for further analysis:
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    print(f"\nUsing record set @id '{main_record_set_id}' for EDA.")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field from the record set for analysis, filter records, normalize data, and, where appropriate, group by other fields. All columns are referenced using their `@id`.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Get a list of numeric fields by inferring from the field definitions
    field_types = {}
    for record_set in metadata.record_sets:
        if record_set.id == main_record_set_id:
            for field in getattr(record_set, 'fields', []):
                field_types[field.id] = getattr(field, 'data_type', None)

    # Pick the first numeric field
    numeric_fields = [fid for fid, dtype in field_types.items() if dtype in ['Float', 'Integer', 'Number'] and fid in df.columns]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        # Only keep rows with non-missing and numeric values
        df_numeric = df[df[numeric_field_id].apply(lambda x: isinstance(x, (int, float, np.number)))]
        threshold = df_numeric[numeric_field_id].mean() if len(df_numeric) > 0 else 10
        filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]

        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_fields = [fid for fid, dtype in field_types.items() if dtype in ['Text', 'String'] and fid in df.columns]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No record sets available for analysis.")

## 5. Visualization

Below, we plot the distribution of the numeric field and (optionally) its grouping by a categorical field. Adjust the field `@id`s as necessary, following the previous sections.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    # Histogram of the numeric field
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")

    # Boxplot by group if a group field exists
    if 'group_field_id' in locals() and group_field_id in df.columns:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=ax[1])
        ax[1].set_title(f"{numeric_field_id} by {group_field_id}")
        ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right')
    else:
        ax[1].axis('off')

    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we successfully loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using `mlcroissant`. We:

- Accessed the dataset schema and metadata
- Inspected available record sets, fields, and their `@id`s
- Loaded records into pandas DataFrames and referenced all entities by `@id`
- Performed basic filtering, normalization, and grouping using referenced field `@id`s
- Visualized distributions for numeric fields.

To continue exploration, modify `@id`s to focus on different record sets or fields and apply your own analysis and visualizations.